# 01 — Config & Utils

Tests `utils/config.py` (YAML loading, typed constants) and `utils/prompts.py` (prompt content sanity checks).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

## 1. Config loading

In [ ]:
import utils.config as cfg

print('=== LLM Models ===')
print('Main model        :', cfg.CLAUDE_MODEL)
print('Classify model    :', cfg.CLAUDE_MODEL_CLASSIFY)
print('Logistics model   :', cfg.CLAUDE_MODEL_LOGISTICS)

print('\n=== Token Limits ===')
print('Classify          :', cfg.CLAUDE_MAX_TOKENS_CLASSIFY)
print('Respond           :', cfg.CLAUDE_MAX_TOKENS_RESPOND)
print('Critique          :', cfg.CLAUDE_MAX_TOKENS_CRITIQUE)
print('Logistics         :', cfg.CLAUDE_MAX_TOKENS_LOGISTICS)

print('\n=== Catalog Agent ===')
print('Max reflection rds:', cfg.MAX_REFLECTION_ROUNDS)
print('Search top_k      :', cfg.CATALOG_SEARCH_TOP_K)
print('Max after filter  :', cfg.CATALOG_MAX_PRODUCTS)

print('\n=== Memory ===')
print('ST max turns      :', cfg.ST_MEMORY_MAX_TURNS)
print('LT embedding model:', cfg.LT_EMBEDDING_MODEL)
print('LT search top_k   :', cfg.LT_SEARCH_TOP_K)

print('\n=== Qdrant ===')
print('Collection        :', cfg.QDRANT_COLLECTION_NAME)
print('Embedding dim     :', cfg.QDRANT_EMBEDDING_DIM)
print('Timeout           :', cfg.QDRANT_TIMEOUT)

print('\n=== Ingest ===')
print('Catalog path      :', cfg.CATALOG_PATH)
print('Ingest model      :', cfg.INGEST_EMBEDDING_MODEL)
print('Batch size        :', cfg.INGEST_BATCH_SIZE)

In [ ]:
# Assert types are what we expect
assert isinstance(cfg.CLAUDE_MODEL, str), 'CLAUDE_MODEL must be str'
assert isinstance(cfg.CLAUDE_MAX_TOKENS_CLASSIFY, int), 'token limit must be int'
assert isinstance(cfg.MAX_REFLECTION_ROUNDS, int), 'reflection rounds must be int'
assert cfg.MAX_REFLECTION_ROUNDS >= 1, 'must have at least 1 reflection round'
assert cfg.CATALOG_SEARCH_TOP_K >= 1, 'search top_k must be positive'
assert cfg.QDRANT_EMBEDDING_DIM == 384, 'all-MiniLM-L6-v2 produces 384-dim vectors'
print('All config assertions passed.')

## 2. Prompts sanity check

In [ ]:
from utils.prompts import (
    ROUTER_SYSTEM_PROMPT,
    CATALOG_SYSTEM_PROMPT,
    REVISE_SYSTEM_PROMPT,
    CRITIC_SYSTEM_PROMPT,
    LOGISTICS_SYSTEM_PROMPT,
)

prompts = {
    'ROUTER_SYSTEM_PROMPT': ROUTER_SYSTEM_PROMPT,
    'CATALOG_SYSTEM_PROMPT': CATALOG_SYSTEM_PROMPT,
    'REVISE_SYSTEM_PROMPT': REVISE_SYSTEM_PROMPT,
    'CRITIC_SYSTEM_PROMPT': CRITIC_SYSTEM_PROMPT,
    'LOGISTICS_SYSTEM_PROMPT': LOGISTICS_SYSTEM_PROMPT,
}

for name, prompt in prompts.items():
    assert isinstance(prompt, str) and len(prompt) > 50, f'{name} looks too short or wrong type'
    print(f'{name}: {len(prompt)} chars  ✓')

In [ ]:
# Spot-check router prompt contains few-shot examples
assert 'SEARCH' in ROUTER_SYSTEM_PROMPT, 'Router prompt must mention SEARCH intent'
assert 'LOGISTICS' in ROUTER_SYSTEM_PROMPT, 'Router prompt must mention LOGISTICS intent'
assert 'PREFERENCE_UPDATE' in ROUTER_SYSTEM_PROMPT, 'Router prompt must mention PREFERENCE_UPDATE intent'

# Critic prompt must mention allergen / safety concepts
assert any(kw in CRITIC_SYSTEM_PROMPT.lower() for kw in ['allergen', 'allergy', 'safety']), \
    'Critic prompt should mention allergen safety'

print('Prompt content assertions passed.')

In [ ]:
# Preview each prompt (first 300 chars)
for name, prompt in prompts.items():
    print(f'\n--- {name} (first 300 chars) ---')
    print(prompt[:300])